# GraphRAG sobre SEC EDGAR — recorrido de extremo a extremo

Construye el grafo (fixture curado + excerpt real de Alphabet), ingiere un 10-K sintético, consulta en los tres modos y reproduce la tabla de evaluación. **Sin llaves de API**: síntesis por plantilla determinista, declarada en cada respuesta (`synthesizer='template'`).

In [2]:
import json, sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from loguru import logger
logger.remove(); logger.add(sys.stderr, level='WARNING')
from src.graph.fixture import build_demo_graph, fixture_summary
g = build_demo_graph()
print(json.dumps(fixture_summary(), indent=1))

{
 "nodes": 271,
 "edges": 443,
 "by_node_type": {
  "Company": 42,
  "Person": 54,
  "Subsidiary": 77,
  "Product": 70,
  "Risk": 13,
  "Market": 15
 },
 "by_edge_type": {
  "CEO_OF": 38,
  "DIRECTOR_OF": 30,
  "SUBSIDIARY_OF": 77,
  "OFFERS_PRODUCT": 70,
  "MENTIONS_RISK": 117,
  "OPERATES_IN": 50,
  "SUPPLIED_BY": 31,
  "COMPETES_WITH": 30
 }
}


## 1. Ingesta: HTML → secciones → chunks → extracción por reglas → MERGE

In [4]:
from src.ingestion.models import Filing
from src.ingestion.parser import html_to_text
from src.ingestion.chunker import split_sections
from src.ingestion.pipeline import ingest_html, load_sample_filings, ingest_sections, SAMPLE_DIR
from src.extraction.rules import RuleBasedExtractor
html = ('<html><body><ix:header><div>us-gaap:hidden</div></ix:header>'
        '<div><span><font>Item</font></span><span><font> 1.</font></span><span><font> Business</font></span></div>'
        '<div>' + 'We design widgets. Our products include WidgetOne and WidgetPro. We compete with Acme Corporation. ' * 8 + '</div>'
        '<div><span><font>Item 1A.</font></span><span> Risk Factors</span></div>'
        '<div>' + 'Our supply chain depends on China. Cybersecurity incidents could harm us. ' * 8 + '</div></body></html>')
print([ (s.item, s.n_chars) for s in split_sections(html_to_text(html)) ])
filing = Filing(ticker='WDGT', cik='0000000009', company_name='Widget Corp', accession='0000000009-25-000001', filing_date='2025-06-30')
report = ingest_html(filing, html, RuleBasedExtractor(), g)
print(report.as_dict())
for f, section in load_sample_filings(ROOT / SAMPLE_DIR):
    print(f.ticker, ingest_sections(f, [section], RuleBasedExtractor(), g).as_dict())

[('1', 791), ('1A', 591)]
{'ticker': 'WDGT', 'accession': '0000000009-25-000001', 'n_sections': 2, 'n_chunks': 2, 'n_entities': 6, 'n_relationships': 5, 'tokens_in': 0, 'tokens_out': 0, 'cost_usd': 0.0, 'latency_ms': 15, 'warnings': []}
GOOGL {'ticker': 'GOOGL', 'accession': '0001652044-26-000018', 'n_sections': 1, 'n_chunks': 5, 'n_entities': 10, 'n_relationships': 9, 'tokens_in': 0, 'tokens_out': 0, 'cost_usd': 0.0, 'latency_ms': 1, 'warnings': []}


## 2. Staleness: nodos con evidencia de más de 365 días

In [6]:
from datetime import date
from src.graph import staleness
report = staleness.mark(g, date(2026, 9, 29))
print(f'{report.stale_count} / {report.total_nodes} nodos stale')
print([ (s.id, s.age_days) for s in report.stale_nodes[:5] ])

49 / 283 nodos stale
[('samsung', 931), ('sk_hynix', 923), ('foxconn', 851), ('panasonic', 824), ('asml', 594)]


## 3. Consulta en los tres modos

In [8]:
from src.engine.graphrag import GraphRAGEngine
engine = GraphRAGEngine(g)
q = 'Which CEOs lead companies that share a supplier with Apple?'
for mode in ('graph', 'vector', 'hybrid'):
    a = engine.answer_sync(q, mode=mode)
    print(f'[{mode:6}] kind={a.classified_kind} ids={a.answer_ids}')
    print('        plan =', [(p.edge_types, p.direction) for p in a.plan], '| latency_ms =', a.metrics.latency_ms)
print()
print(engine.answer_sync('How many S&P 500 companies mention China exposure as a risk?').answer)
print(engine.answer_sync('Which products does Alphabet offer?').answer_ids)

[graph ] kind=multi_hop ids=['cristiano_amon', 'hock_tan', 'jensen_huang', 'jim_farley', 'lip_bu_tan', 'lisa_su', 'mark_zuckerberg', 'mary_barra', 'sundar_pichai']
        plan = [(['SUPPLIED_BY'], 'out'), (['SUPPLIED_BY'], 'in'), (['CEO_OF'], 'in')] | latency_ms = 1
[vector] kind=hybrid ids=['tim_cook']
        plan = [] | latency_ms = 52
[hybrid] kind=multi_hop ids=['cristiano_amon', 'hock_tan', 'jensen_huang', 'jim_farley', 'lip_bu_tan', 'lisa_su', 'mark_zuckerberg', 'mary_barra', 'sundar_pichai']
        plan = [(['SUPPLIED_BY'], 'out'), (['SUPPLIED_BY'], 'in'), (['CEO_OF'], 'in')] | latency_ms = 0

count = 13 — Advanced Micro Devices, Inc., Apple Inc., The Boeing Company, Broadcom Inc., Ford Motor Company, General Motors Company, Intel Corporation, Micron Technology, Inc. (+5 more)
['prod_android', 'prod_chrome', 'prod_gemini', 'prod_gmail', 'prod_google_cloud', 'prod_google_drive', 'prod_google_gemini', 'prod_google_maps', 'prod_google_photos', 'prod_google_play', 'prod_google_se

## 4. Evaluación: Vector RAG vs GraphRAG vs Hybrid vs ablación

In [10]:
import tempfile
from src.evaluation.runner import run_all, render_results_md
with tempfile.TemporaryDirectory() as tmp:
    run_path = run_all(build_demo_graph(), ROOT / 'eval' / 'gold.jsonl', ROOT / 'eval' / 'extraction_gold.jsonl', Path(tmp), 'notebook')
    md = render_results_md(run_path)
print('\n'.join(l for l in md.splitlines() if l.startswith('|'))[:1600])

| Sistema | Lookup Hit | 2-hop Hit | 3-hop Hit | Aggregation Hit | Faithfulness |
|---|---|---|---|---|---|
| Vector RAG (TF-IDF, mismo corpus) | 86.7 % | 30.0 % | 0.0 % | 35.0 % | pendiente (requiere ANTHROPIC_API_KEY) |
| GraphRAG (este sistema) | 100.0 % | 100.0 % | 100.0 % | 100.0 % | pendiente (requiere ANTHROPIC_API_KEY) |
| Hybrid: Graph + Vector | 100.0 % | 100.0 % | 100.0 % | 85.0 % | pendiente (requiere ANTHROPIC_API_KEY) |
| Ablation: GraphRAG sin traversal multi-hop (max_hops=1) | 100.0 % | 30.0 % | 0.0 % | 95.0 % | pendiente (requiere ANTHROPIC_API_KEY) |
| Sistema | Hit global | Precisión media (ids) | Routing acc. | p50 ms | p95 ms | Tokens in/out | Costo USD |
|---|---|---|---|---|---|---|---|
| Vector RAG (TF-IDF, mismo corpus) | 42.0 % | 40.1 % | 20.0 % | 17 | 19 | 0/0 | 0.0000 |
| GraphRAG (este sistema) | 100.0 % | 100.0 % | 96.0 % | 0 | 0 | 0/0 | 0.0000 |
| Hybrid: Graph + Vector | 97.0 % | 97.0 % | 96.0 % | 0 | 0 | 0/0 | 0.0000 |
| Ablation: GraphRAG sin traversal

## 5. Qué falta para el sistema completo

* `ANTHROPIC_API_KEY` → `ClaudeExtractor` sobre 100 × 3 filings reales y `ClaudeSynthesizer` con citas; faithfulness con la rúbrica de `src/evaluation/metrics.py`.
* Salida de red a `sec.gov` → `scripts/download_data.py`.
* Docker → `GRAPH_BACKEND=neo4j` con `docker compose up`.